In [ ]:
# ============================================================
# EXPERIMENT 4A: KNN CLASSIFICATION FROM SCRATCH
# ============================================================
import numpy as np
from collections import Counter

class KNNClassifier:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train, self.y_train = np.array(X), np.array(y)

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_one(self, x):
        distances = [(self.euclidean_distance(x, x_t), y_t) for x_t, y_t in zip(self.X_train, self.y_train)]
        distances.sort(key=lambda item: item[0])
        k_nearest_labels = [label for _, label in distances[:self.k]]
        return Counter(k_nearest_labels).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self.predict_one(x) for x in np.array(X)])

    def accuracy(self, y_true, y_pred):
        return np.mean(np.array(y_true) == np.array(y_pred))

# Example Dataset (X1=Study Hours, X2=Attendance, 0=Fail, 1=Pass)
X_train = np.array([[1, 60], [2, 65], [3, 70], [4, 75], [5, 80], [6, 85]])
y_train = np.array([0, 0, 1, 1, 1, 1])
X_test = np.array([[2.5, 68], [4.5, 78], [5.5, 82]])
y_test = np.array([0, 1, 1])

# Create, Train, and Predict
knn = KNNClassifier(k=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Display Results
print(f"Actual Labels : {y_test}\nPredicted Labels : {y_pred}")
acc = knn.accuracy(y_test, y_pred)
print(f"Accuracy : {acc}\nAccuracy (%) : {acc * 100}")

In [ ]:
# ============================================================
# EXPERIMENT 4B: KNN REGRESSION FROM SCRATCH
# ============================================================
import numpy as np

class KNNRegressor:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train, self.y_train = np.array(X, dtype=float), np.array(y, dtype=float)

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_one(self, x):
        distances = sorted([(self.euclidean_distance(x, x_t), y_t) for x_t, y_t in zip(self.X_train, self.y_train)])
        return np.mean([y_t for _, y_t in distances[:self.k]])

    def predict(self, X):
        return np.array([self.predict_one(x) for x in np.array(X, dtype=float)])

    def mse(self, y_true, y_pred):
        return np.mean((np.array(y_true) - np.array(y_pred)) ** 2)

    def mae(self, y_true, y_pred):
        return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

    def r2_score(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))

# Example Dataset (X1=House Area, X2=Bedrooms, Y=Price in Lakhs)
X_train = np.array([[800, 2], [1000, 2], [1200, 3], [1400, 3], [1600, 4], [1800, 4]])
y_train = np.array([40, 50, 60, 70, 80, 90])
X_test = np.array([[1100, 2], [1500, 3], [1700, 4]])
y_test = np.array([55, 75, 85])

# Create, Train, and Predict
knn = KNNRegressor(k=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Display Results
print(f"Actual Values : {y_test}\nPredicted Values : {np.round(y_pred, 2)}")

# Evaluation
mse = knn.mse(y_test, y_pred)
mae = knn.mae(y_test, y_pred)
r2 = knn.r2_score(y_test, y_pred)

print(f"\nPerformance\n------------------\nMSE : {round(mse, 4)}\nMAE : {round(mae, 4)}\nR2 : {round(r2, 4)}")

In [ ]:
# ============================================================
# EXPERIMENT 4C: KNN CLASSIFICATION USING SCIKIT-LEARN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load Dataset and Create DataFrame
iris = load_iris()
X, y = iris.data, iris.target
feature_names, target_names = iris.feature_names, iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Species"] = df["Target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

print(f"Dataset Shape:\n{df.shape}")
print(f"\nFirst Five Records:\n{df.head()}")

# Dataset Information
print(f"\nFeature Names:\n{feature_names}")
print(f"\nTarget Classes:\n{target_names}")
print(f"\nClass Distribution:\n{df['Species'].value_counts()}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Feature Scaling, Model Creation, and Training
scaler = StandardScaler()
X_train_scaled, X_test_scaled = scaler.fit_transform(X_train), scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5, weights="uniform", metric="euclidean")
knn.fit(X_train_scaled, y_train)

# Prediction and Results
y_pred = knn.predict(X_test_scaled)
results = pd.DataFrame({
    "Actual": y_test, "Predicted": y_pred,
    "Actual Species": [target_names[i] for i in y_test],
    "Predicted Species": [target_names[i] for i in y_pred]
})
print(f"\nActual vs Predicted:\n{results}")

# Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"\n=================================\nKNN CLASSIFICATION PERFORMANCE\n=================================")
print(f"Accuracy : {round(accuracy, 4)}\nAccuracy (%) : {round(accuracy * 100, 2)}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks(range(len(target_names)), target_names)
plt.yticks(range(len(target_names)), target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Experiment 4D: KNN Regression Using Scikit-learn
# (Offline-safe version using load_diabetes)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------- Step 1: Load bundled dataset (NO network needed) ----------
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame.copy()              # features + 'target' column
feature_names = diabetes.feature_names  # list of 10 feature names

# ---------- Step 2: Inspect ----------
print("Dataset Shape:", df.shape)
print("\nFirst Five Records:")
print(df.head())
print("\nFeature Names:", feature_names)
print("\nDataset Description:")
print(df.describe())

# ---------- Step 3: Separate features / target ----------
X = df[feature_names].values
y = df["target"].values

# ---------- Step 4: Train-test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples :", X_test.shape[0])

# ---------- Step 5: Feature scaling ----------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ---------- Step 6: KNN model ----------
knn = KNeighborsRegressor(n_neighbors=5, weights="uniform", metric="euclidean")
knn.fit(X_train_scaled, y_train)

# ---------- Step 7: Predict ----------
y_pred = knn.predict(X_test_scaled)

results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
print("\nActual vs Predicted:")
print(results.head(10))

# ---------- Step 8: Metrics ----------
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("\n===================================")
print("KNN REGRESSION PERFORMANCE")
print("===================================")
print("MAE  :", round(mae, 4))
print("MSE  :", round(mse, 4))
print("RMSE :", round(rmse, 4))
print("R2   :", round(r2, 4))

# ---------- Step 9: Actual vs Predicted plot ----------
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, color="red", alpha=0.5, label="Predicted Values")
mn = min(y_test.min(), y_pred.min())
mx = max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx], color="blue", linewidth=2, label="Perfect Prediction")
plt.xlabel("Actual Disease Progression")
plt.ylabel("Predicted Disease Progression")
plt.title("KNN Regression: Actual vs Predicted")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# ---------- Step 10: Error distribution ----------
errors = y_test - y_pred
plt.figure(figsize=(8, 6))
plt.hist(errors, bins=30)
plt.xlabel("Prediction Error"); plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")
plt.grid(True); plt.tight_layout(); plt.show()